# ScamRoad — Autonomous Cloud GPU Training, Model Comparison & Inference Pipeline

**Repository**: [github.com/Priya-Ranjan-0201/potholes](https://github.com/Priya-Ranjan-0201/potholes)  
**Compute Environment**: Google Colab GPU (Tesla T4 / V100 / A100 with CUDA)  
**Persistent Storage**: Google Drive (`/content/drive/MyDrive/ScamRoad`)  
**Architectures Compared**: YOLOv8n (Nano), YOLOv8s (Small), YOLOv8n-seg (Segmentation), RT-DETR-l (Transformer)  
**Single-Source Taxonomy**: `pothole`, `crack_longitudinal`, `crack_transverse`, `crack_alligator`


## Step 1: Hardware & System Resource Audit
Verify the assigned GPU, VRAM, and system memory before launching any heavy operations.

In [ ]:
# 1. GPU & System Resource Audit
!nvidia-smi

import psutil
import shutil
import torch

ram = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print('=' * 65)
print(f'PyTorch Version  : {torch.__version__}')
print(f'CUDA Available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device       : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'VRAM Capacity    : {vram:.2f} GB')
    print(f'CUDA Version     : {torch.version.cuda}')
else:
    print('[WARNING] No CUDA GPU detected! Switch Runtime -> Change runtime type -> T4 GPU')
print(f'Colab RAM Total  : {ram.total / (1024**3):.2f} GB (Available: {ram.available / (1024**3):.2f} GB)')
print(f'Colab Disk Free  : {disk.free / (1024**3):.2f} GB / {disk.total / (1024**3):.2f} GB')
print('=' * 65)


## Step 2: Mount Persistent Google Drive Storage
Mount Google Drive to persist checkpoints, trained weights, logs, and exports across Colab disconnects.

In [ ]:
# 2. Google Drive Mount & Workspace Directory Initialization
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/ScamRoad')
drive_root.mkdir(parents=True, exist_ok=True)

for folder in ['datasets', 'checkpoints', 'weights', 'exports', 'runs', 'logs', 'comparisons']:
    (drive_root / folder).mkdir(exist_ok=True)

print(f'[SUCCESS] Persistent Google Drive storage active at: {drive_root}')
!ls -lh /content/drive/MyDrive/ScamRoad


## Step 3: Clone or Pull ScamRoad Project Repository
Pulls the latest verified codebase from GitHub repository `Priya-Ranjan-0201/potholes`.

In [ ]:
# 3. Clone or pull repo
import os, sys
from pathlib import Path

repo_path = Path('/content/potholes')
if not repo_path.exists():
    !git clone https://github.com/Priya-Ranjan-0201/potholes.git /content/potholes
else:
    !cd /content/potholes && git pull origin main

if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

# Set working directory to repo root
os.chdir(str(repo_path))
print('[SUCCESS] Project workspace ready at:', repo_path.resolve())


## Step 4: Install Cloud Dependencies
Install Ultralytics YOLOv8, ONNX, and OpenCV acceleration libraries.

In [ ]:
# 4. Pip installation
!pip install -q ultralytics onnx onnxruntime-gpu pyyaml albumentations opencv-python psutil


## Step 5: Configure Kaggle API Credentials
Authenticates Kaggle directly from Colab Secrets (`priyaranjanng`, `KAGGLE_KEY`, or `KAGGLE_API_TOKEN`).

In [ ]:
# 5. Kaggle Authentication
import contextlib, json, os
from pathlib import Path

def clean_token(raw_token):
    if not raw_token:
        return None
    token = raw_token.strip()
    if (token.startswith('export ') and '=' in token) or ('=' in token and not token.startswith('{')):
        token = token.split('=', 1)[1].strip()
    return token.strip('"\'\r\n ')

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_file = kaggle_dir / 'kaggle.json'
access_token_file = kaggle_dir / 'access_token'

try:
    from google.colab import userdata
    user = None
    raw_key = None
    with contextlib.suppress(Exception): user = userdata.get('KAGGLE_USERNAME')
    with contextlib.suppress(Exception): raw_key = userdata.get('KAGGLE_KEY')
    with contextlib.suppress(Exception):
        if not raw_key: raw_key = userdata.get('KAGGLE_API_TOKEN')
    for candidate in ['priyaranjanng', 'priyaranja', 'priyaranjan', 'kaggle']:
        if not raw_key:
            with contextlib.suppress(Exception): raw_key = userdata.get(candidate)
    key = clean_token(raw_key)
    username = user or 'priyaranjanng'
    if key:
        access_token_file.write_text(key, encoding='utf-8')
        access_token_file.chmod(0o600)
        kaggle_file.write_text(json.dumps({'username': username, 'key': key}), encoding='utf-8')
        kaggle_file.chmod(0o600)
        os.environ['KAGGLE_USERNAME'] = username
        os.environ['KAGGLE_KEY'] = key
        os.environ['KAGGLE_API_TOKEN'] = key
        print(f'[SUCCESS] Kaggle credentials configured from Colab Secrets (user: {username})!')
except Exception as e:
    print('[INFO] Colab secrets check:', e)

!kaggle --version


## Step 6: Dataset Download & Ingestion Pipeline
Downloads road damage datasets (MakeML Potholes and RDD2022) using Kaggle API,
converts annotations to standard YOLO format, audits class balances, and generates leakage-free sequence splits.

In [ ]:
# 6. Download Road Defect Datasets from Kaggle
from pathlib import Path

data_raw = Path('/content/data/raw')
data_raw.mkdir(parents=True, exist_ok=True)

# Option A: MakeML Potholes (Fast, clean Pascal VOC)
!kaggle datasets download -d larxel/pothole-detection -p /content/data/raw/potholes --unzip

# Convert VOC annotations to normalized YOLO format
!python -m ml.data.convert --input-dir /content/data/raw/potholes/annotations --images-dir /content/data/raw/potholes/images --out-dir /content/data/processed/potholes

# Audit dataset (pHash deduplication, corrupt checks, class distributions)
!python -m ml.data.audit --images-dir /content/data/processed/potholes/images --labels-dir /content/data/processed/potholes/labels

# Split into 70/15/15 sequence-aware splits
!python -m ml.data.split --images-dir /content/data/processed/potholes/images --labels-dir /content/data/processed/potholes/labels --out-dir /content/data/splits --train-ratio 0.70 --val-ratio 0.15 --test-ratio 0.15

print('[SUCCESS] Dataset ready for model training.')


## Step 7: Train Model A — YOLOv8n (Nano Baseline)
Ultra-lightweight (3.2M params) edge detector optimized for mobile devices (OPPO K13 / Android) and SBCs.
Checkpoints are saved directly to Google Drive so no epoch progress is lost.

In [ ]:
# 7. Train YOLOv8n
from ultralytics import YOLO
from pathlib import Path

drive_runs = Path('/content/drive/MyDrive/ScamRoad/runs')
dataset_cfg = Path('/content/data/splits/dataset.yaml')

model_n = YOLO('yolov8n.pt')
results_n = model_n.train(
    data=str(dataset_cfg),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(drive_runs),
    name='model_yolov8n',
    save=True,
    exist_ok=True,
    verbose=True
)
print('[SUCCESS] YOLOv8n training finished.')


## Step 8: Train Model B — YOLOv8s (Small Detector)
Higher capacity (11.2M params) for superior recall on fine longitudinal and transverse cracks.

In [ ]:
# 8. Train YOLOv8s
model_s = YOLO('yolov8s.pt')
results_s = model_s.train(
    data=str(dataset_cfg),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(drive_runs),
    name='model_yolov8s',
    save=True,
    exist_ok=True,
    verbose=True
)
print('[SUCCESS] YOLOv8s training finished.')


## Step 9: Train Model C — YOLOv8n-seg (Instance Segmentation)
Predicts pixel-level defect masks alongside bounding boxes to compute exact road defect surface area.

In [ ]:
# 9. Train YOLOv8n-seg
model_seg = YOLO('yolov8n-seg.pt')
results_seg = model_seg.train(
    data=str(dataset_cfg),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(drive_runs),
    name='model_yolov8n_seg',
    save=True,
    exist_ok=True,
    verbose=True
)
print('[SUCCESS] YOLOv8n-seg training finished.')


## Step 10: Train Model D — RT-DETR-l (Real-Time Transformer)
End-to-end detection transformer removing NMS thresholds and excelling on dense crack clusters.

In [ ]:
# 10. Train RT-DETR-l
model_rtdetr = YOLO('rtdetr-l.pt')
results_rtdetr = model_rtdetr.train(
    data=str(dataset_cfg),
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    project=str(drive_runs),
    name='model_rtdetr_l',
    save=True,
    exist_ok=True,
    verbose=True
)
print('[SUCCESS] RT-DETR-l training finished.')


## Step 11: Multi-Model Architecture Comparison & Benchmark
Evaluates all 4 models on the validation set, benchmarks latency, and generates a structured Markdown comparison table.

In [ ]:
# 11. Run Multi-Model Comparison Harness
comparison_dir = Path('/content/drive/MyDrive/ScamRoad/comparisons')

!python -m ml.evaluation.compare_models \
    --data /content/data/splits/dataset.yaml \
    --models \
        YOLOv8n=/content/drive/MyDrive/ScamRoad/runs/model_yolov8n/weights/best.pt \
        YOLOv8s=/content/drive/MyDrive/ScamRoad/runs/model_yolov8s/weights/best.pt \
        YOLOv8n-seg=/content/drive/MyDrive/ScamRoad/runs/model_yolov8n_seg/weights/best.pt \
        RT-DETR-l=/content/drive/MyDrive/ScamRoad/runs/model_rtdetr_l/weights/best.pt \
    --out-dir /content/drive/MyDrive/ScamRoad/comparisons \
    --device 0

# Display Markdown Comparison Report
from IPython.display import Markdown, display
report_text = (comparison_dir / 'model_comparison.md').read_text()
display(Markdown(report_text))


## Step 12: ONNX INT8 Quantization & Mobile Edge Export
Exports the top-performing model to ONNX INT8 for edge deployment on Android and low-power SBCs.

In [ ]:
# 12. Export & Quantize Best Model
import shutil
from pathlib import Path
from onnxruntime.quantization import quantize_dynamic, QuantType

best_pt = drive_runs / 'model_yolov8n' / 'weights' / 'best.pt'
exports_dir = Path('/content/drive/MyDrive/ScamRoad/exports')
exports_dir.mkdir(parents=True, exist_ok=True)

if best_pt.exists():
    best_model = YOLO(str(best_pt))
    onnx_file = best_model.export(format='onnx', imgsz=640, simplify=True)
    fp32_dest = exports_dir / 'scamroad_best_fp32.onnx'
    shutil.copy(onnx_file, fp32_dest)

    int8_dest = exports_dir / 'scamroad_best_int8.onnx'
    quantize_dynamic(
        model_input=str(fp32_dest),
        model_output=str(int8_dest),
        weight_type=QuantType.QUInt8
    )
    print(f'[SUCCESS] FP32 ONNX : {fp32_dest} ({fp32_dest.stat().st_size / (1024**2):.2f} MB)')
    print(f'[SUCCESS] INT8 ONNX : {int8_dest} ({int8_dest.stat().st_size / (1024**2):.2f} MB)')


## Step 13: Static Image & Real-Time Video Defect Inference
Runs `scripts/infer_media.py` on test images and video with bounding boxes, HUD overlay, and relative damage area %.

In [ ]:
# 13. Test Image & Video Inference
!python scripts/infer_media.py \
    --source /content/data/splits/images/val \
    --weights /content/drive/MyDrive/ScamRoad/runs/model_yolov8n/weights/best.pt \
    --output-dir /content/drive/MyDrive/ScamRoad/inference_demo \
    --conf 0.25

print('[SUCCESS] Inference demonstration complete. Results saved in Google Drive.')
!ls -lh /content/drive/MyDrive/ScamRoad/inference_demo


## Step 14: Google Drive Persistent Inventory
Lists all finalized weights, ONNX models, benchmark reports, and inference media stored in Google Drive.

In [ ]:
# 14. Artifact Inventory
!ls -R /content/drive/MyDrive/ScamRoad
print('\n[COMPLETE] ScamRoad Cloud GPU Pipeline completed successfully.')
